In [109]:
from datetime import datetime, timedelta
import os
import numpy as np
import pandas as pd

import Strategies.Autotrader.enumerate as ENUM
from Utilities.Storage import get_curr_storage_path
from Strategies.Autotrader.TP_api import TP_api

### Enviroment

In [110]:
ENV = 'prod'
cls = TP_api(ENV)
cls.token

'83148b98-3bbf-4a04-ada3-6f7a5af02f1d'

### Strategy config

#### Calculate product id for desired month product

In [111]:
month_item_id_to_month = lambda x: (x-241)%12+1

In [112]:
def month_to_item_id(month):
    return 241 + (month - 1) + 12 * ((256 - 241) // 12)

In [113]:
from SynthSpread.spreadviewer_class import SpreadSingle
today = datetime.now().date() + timedelta(days=1)
dates = pd.date_range(today, today, freq='B')
spread_class = SpreadSingle(['de'], ['m'], [1], [], ['eex'])
product_date = spread_class.product_dates(dates, 2, tn_bool=True)
item_id = month_to_item_id(product_date[0][0].month)
item_id

258

In [ ]:

algo_id = "obs"
algo_caption = "OBS DE M1"
exchange_id = "TRAYPORT"
package_name= "observer_strategy_v0_1"
instrument_id = ENUM.InstrumentID.DE_BASE_EEX
sequence_id = ENUM.SequenceID.M
product_id = f'10000104_{item_id}'#251 nov
brokers_list = [ENUM.Broker._EEX,
                    ENUM.Broker._GFI,
                    ENUM.Broker._ICAP,
                    ENUM.Broker._42FS,
                    ENUM.Broker._TFS,
                    ENUM.Broker._SPEC,
                    ENUM.Broker._GRFN]
#test env vars
product_id = '10000104_265' # 254februar
brokers_list = ['37', '38'] # BRO 2, 3


config = {
    "internal_number": algo_id,
    "caption": algo_caption,
    "exchange": exchange_id,
    "package_name": package_name,
    "instrument_ids": [instrument_id],
    "product_ids": [product_id],
    "broker_list": brokers_list,
}

#### Limits

Change the `value` to desired limit and script will set unused products to 0.

This will also set allowed price boundaries `_MAXIMUM_PRICE`, `_MINIMUM_PRICE`.

In [126]:
_MAXIMUM_PRICE = 200.0
_MINIMUM_PRICE = 10.0
limits = {
    "limits_per_sequence": {
        "maximum_purchase_price": {},
        "minimum_sales_price": {},
        "maximum_purchase_volume": {},
        "maximum_sales_volume": {}
    }
}
for product in ["10000100",
                "10000101", 
                "10000102", 
                "10000103", 
                "10000104", 
                "10000105", 
                "10000106"]:
    if product not in product_id:
        value = 0
    else:
        value = 3
    limits["limits_per_sequence"]["maximum_purchase_price"][product] = _MAXIMUM_PRICE
    limits["limits_per_sequence"]["minimum_sales_price"][product] = _MINIMUM_PRICE
    limits["limits_per_sequence"]["maximum_sales_volume"][product] = value
    limits["limits_per_sequence"]["maximum_purchase_volume"][product] = value

limits
    

{'limits_per_sequence': {'maximum_purchase_price': {'10000100': 200.0,
   '10000101': 200.0,
   '10000102': 200.0,
   '10000103': 200.0,
   '10000104': 200.0,
   '10000105': 200.0,
   '10000106': 200.0},
  'minimum_sales_price': {'10000100': 10.0,
   '10000101': 10.0,
   '10000102': 10.0,
   '10000103': 10.0,
   '10000104': 10.0,
   '10000105': 10.0,
   '10000106': 10.0},
  'maximum_purchase_volume': {'10000100': 0,
   '10000101': 0,
   '10000102': 0,
   '10000103': 0,
   '10000104': 3,
   '10000105': 0,
   '10000106': 0},
  'maximum_sales_volume': {'10000100': 0,
   '10000101': 0,
   '10000102': 0,
   '10000103': 0,
   '10000104': 3,
   '10000105': 0,
   '10000106': 0}}}

### Create strategy

Check active strategies

In [127]:
cls.deactivate_strategy(algo_id)
cls.delete_strategy(algo_id)

<Response [200]>

In [128]:
print(cls.create_strategy(config))

{'product_ids': ['10000104_258'], 'broker_list': ['1441', '2', '4', '8', '7', '5', '3'], 'stop_loss_margin': 0.4, 'hard_stop_loss': 1.0, 'trd_gap': 0.1, 'make_profit_margin': 0.25, 'loss_making_thold': 0.1, 'ql_max': 0.7, 'makeagg_ratio_thold': 0.6, 'burnout_period': 7.0, 'bid_ask': 0.3, 'max_position': 1.0, 'lead_closing': False, 'thold_dense': 0.3, 'thold_sparse': 0.13, 'internal_number': 'spb-de-m-1', 'caption': 'SPB DE M1 v0.9', 'exchange': 'TRAYPORT', 'active': False, 'instrument_ids': ['10641710'], 'package_name': 'spbmark_strategy_v09', 'halted': False, 'halt_reason': ''}


In [129]:
result = cls.set_limits(limits, algo_id=algo_id)
result

{'strategy_id': 'spb-de-m-1',
 'exchange_id': 'TRAYPORT',
 'limits_per_sequence': {'maximum_purchase_price': {'10000100': 200.0,
   '10000101': 200.0,
   '10000102': 200.0,
   '10000103': 200.0,
   '10000104': 200.0,
   '10000105': 200.0,
   '10000106': 200.0},
  'maximum_purchase_volume': {'10000100': 0,
   '10000101': 0,
   '10000102': 0,
   '10000103': 0,
   '10000104': 3,
   '10000105': 0,
   '10000106': 0},
  'maximum_sales_volume': {'10000100': 0,
   '10000101': 0,
   '10000102': 0,
   '10000103': 0,
   '10000104': 3,
   '10000105': 0,
   '10000106': 0},
  'minimum_sales_price': {'10000100': 10.0,
   '10000101': 10.0,
   '10000102': 10.0,
   '10000103': 10.0,
   '10000104': 10.0,
   '10000105': 10.0,
   '10000106': 10.0}},
 'limits_per_sequence_item': {'maximum_purchase_price': {},
  'maximum_purchase_volume': {},
  'maximum_sales_volume': {},
  'minimum_sales_price': {}},
 'update_time': '2025-05-11T13:04:46.746000'}